# Introduction

In this notebook, I will work on a final modelisation to infer result for my thesis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from functionsFolder.config import text_data, biometric_data, adv_to_del, team_to_del, id, useless, index, to_keep
from functionsFolder.preProcessingAuto import preprocess_working_df
from functionsFolder.modelAutomation import get_features_and_target, clean_features, evaluate_models, prepare_features
from functionsFolder.optimisationModel import select_features, optimized_training
from sklearn.feature_selection import SelectPercentile
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
from time import time
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
import joblib
import shap
from matplotlib.colors import LinearSegmentedColormap

##Different Paths
HOME = r"C:\Users\Utilisateur\Desktop\Master ULB\Mémoire"
W_DB = r"\Database\Working db"
STACK = r"\Thesis - Code\database\Database Updates\Database stack"

In [ ]:
test_idf = pd.read_excel(r"C:\Users\Utilisateur\Desktop\Master ULB\Mémoire\Thesis - Code\tf_idf_df.xlsx")

In [ ]:
team_to_avg = [
    col for col in test_idf.columns
    if ('team' in col or 'opp' in col) and 'text_' not in col and '%' not in col
]  #Creating a list with the columns to be divided

team_to_avg.remove('G_team_features_context') #Removing the number of games

test_idf[team_to_avg] = test_idf[team_to_avg].div(test_idf['G_team_features_context'], axis=0) #Dividing all the columns by the number of games

## To delete

In [26]:
dico_features_tfidf = {
    'Without' : ['Weight_features',
 'per_40_FG%_features',
 'per_40_3P%_features',
 'per_40_2P%_features',
 'per_40_FT%_features',
 'per_40_ORB_features',
 'per_40_TRB_features',
 'per_40_STL_features',
 'per_40_TOV_features',
 'advanced_PER_features',
 'advanced_3PAr_features',
 'advanced_FTr_features',
 'advanced_ORB%_features',
 'advanced_AST%_features',
 'advanced_STL%_features',
 'advanced_BLK%_features',
 'advanced_WS/40_features',
 'advanced_OBPM_features',
 'advanced_BPM_features'],
'With' : 
['Height_features',
 'Weight_features',
 'text_game_features_context',
 'per_40_FG_features',
 'per_40_FG%_features',
 'per_40_3P_features',
 'per_40_3PA_features',
 'per_40_3P%_features',
 'per_40_2PA_features',
 'per_40_2P%_features',
 'per_40_FT_features',
 'per_40_FTA_features',
 'per_40_FT%_features',
 'per_40_ORB_features',
 'per_40_DRB_features',
 'per_40_TRB_features',
 'per_40_AST_features',
 'per_40_STL_features',
 'per_40_BLK_features',
 'per_40_TOV_features',
 'per_40_PF_features',
 'G_team_features_context',
 'W-L%_features_context',
 'W.2_features_context',
 'Tm._features_context',
 'Opp._features_context',
 'FGA_team_features_context',
 '3PA_team_features_context',
 'FT_team_features_context',
 'FT%_team_features_context',
 'TRB_team_features_context',
 'PF_team_features_context',
 'FG_opp_features_context',
 'FGA_opp_features_context',
 '3P%_opp_features_context',
 'STL_opp_features_context',
 'PF_opp_features_context',
 'advanced_PER_features',
 'advanced_TS%_features',
 'advanced_3PAr_features',
 'advanced_FTr_features',
 'advanced_ORB%_features',
 'advanced_DRB%_features',
 'advanced_AST%_features',
 'advanced_OWS_features',
 'advanced_WS_features',
 'advanced_WS/40_features',
 'advanced_OBPM_features',
 'advanced_DBPM_features',
 'advanced_BPM_features']
}

## Add index

We need index for latter analysis

In [8]:
base_df = pd.read_excel(HOME + STACK + r"\looping_df_1746884524.8275206.xlsx")
RECOVERY_OPP = HOME + W_DB + r"\Features\Team data\27-4-2025_team.xlsx"

In [9]:
#add the mean of every position in the advance part and scouting evaluation part
base_df[to_keep['adv']] = base_df[to_keep['adv']].fillna(base_df.groupby('Pos_x_features')[to_keep['adv']].transform('median'))
base_df[to_keep['scouting_reports']] = base_df[to_keep['scouting_reports']].fillna(base_df.groupby('Pos_x_features')[to_keep['scouting_reports']].transform('median'))
processed_df = preprocess_working_df(base_df, recovery_file=RECOVERY_OPP)
#Get a copy of the dataset
working_df = processed_df.copy()
# Get initial features and targets
raw_features, target = get_features_and_target(working_df) #Get the features and target columns' name in separate lists


# Step 2: Prepare feature list (optionally re-adding 'adv' and 'scouting_reports')
include_subsets = ['per_40', 'team', 'opp', 'adv', 'scouting_reports']
features, included_types = prepare_features( #prepare the features' space
    base_features=raw_features,
    df=working_df,
    useless_list=useless, #list of features that we do not use 
    feature_dict=to_keep, #dictionnary of the features that we keep 
    include_keys=include_subsets #the keys to the dictionnary above
)
#Change the height from feet to inches 
working_df['Height_features'] = working_df['Height_features'].apply(
    lambda x: int(x.split('-')[0]) * 12 + int(x.split('-')[1])
)

#We keep only the players that where drafted after 2009 (60 players for 15 years = 900 max)
df = working_df[working_df['draft_season_features'] > 8].copy()
i = 1
df['med_tresh'] = (df[f'WS/48-{i}_target'] > 0).astype(int)

#We keep context variables in an list
keywords = ['team', 'opp', 'text']
filtered_list = [element for element in features if not any(keyword in element for keyword in keywords)]

2025-05-15 18:17:23,095 - INFO - Starting full preprocessing pipeline...
2025-05-15 18:17:23,097 - INFO - Creating dummy variables...
2025-05-15 18:17:23,097 - INFO - After dummy variable creation: shape = (1253, 389)
2025-05-15 18:17:23,097 - INFO - Categorizing and one-hot encoding categorical features...
2025-05-15 18:17:23,121 - INFO - After categorization and one-hot encoding: shape = (1253, 405)
2025-05-15 18:17:23,122 - INFO - Processing unstructured text data...
2025-05-15 18:17:26,786 - INFO - After TF-IDF vectorization: shape = (1253, 8291)
2025-05-15 18:17:26,836 - INFO - After after TF-IDF merge: shape = (1253, 8696)
2025-05-15 18:17:26,841 - INFO - Imputing opponent columns based on team data...
2025-05-15 18:17:29,012 - INFO - After imputation using team data: shape = (1253, 8696)
2025-05-15 18:17:29,014 - INFO - Imputing opponent columns using recovery data...
2025-05-15 18:17:36,095 - INFO - After imputation using recovery data: shape = (1253, 8696)
2025-05-15 18:17:36,

In [13]:
test_idf.set_index(df['player_id'])

,Height_features,Weight_features,dummy_hs_ranking_features_context,dummy_coll_awards_features_context,dummy_mock_draft_features_context,dummy_scouting_reports_features_context,mock_draft_1-25_features_context,mock_draft_26-50_features_context,mock_draft_51-75_features_context,mock_draft_76-100_features_context,...,Strength2_features_context,Quickness_features_context,Leadership_features_context,Jump Shot_features_context,NBA Ready_features_context,Rebounding_features_context,Potential_features_context,Post Skills_features_context,Intangibles_features_context,med_tresh
player_id,,,,,,,,,,,,,,,,,,,,,
aarongordon_15,81,225,1,1,1,1,1,0,0,0,...,7,8,8,6,8.0,7,8.0,6,9,1
aaronharrison_16,78,212,1,0,1,1,0,0,1,0,...,8,7,6,8,7.0,7,7.0,7,7,0
aaronhenry_22,78,210,0,0,1,1,0,0,1,0,...,8,8,7,7,8.0,6,7.0,7,7,0
aaronholiday_19,73,185,1,1,1,1,0,1,0,0,...,8,8,8,8,8.0,8,7.0,7,8,1
aaronjackson_18,80,215,0,0,0,0,0,0,0,0,...,7,8,8,8,8.0,8,7.0,7,8,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
zhairesmith_19,77,195,0,0,1,1,1,0,0,0,...,7,8,7,7,7.0,9,7.0,7,7,1
ziairewilliams_22,80,185,1,0,1,1,1,0,0,0,...,7,8,7,8,7.0,8,9.0,8,8,1
zionwilliamson_20,79,285,1,1,1,1,1,0,0,0,...,9,9,9,7,8.0,8,9.0,7,9,1


# TF-IDF Pipeline

In [ ]:
features = [key for key in test_idf.columns if 'feature' in key] #All the features in a list 
filtered_list = [key for key in features if 'context' not in key] #The features without the contextual one in another

In [ ]:
#Configuration
config = {
    'test_size': 0.2,
    'n_folds': 5, #CV=5
    'n_iter' : 200, 
    'scoring': 'roc_auc', #Optimize the AUC
    'features_max': 100,
    'features_sets' : { #To automatically change the dataset from contextual to not
        'With': features,
        #'Without' : filtered_list
    }
}

dico_features_tfidf = {} #dictionnary for to features retain in select_features function

#X & y
y = test_idf['med_tresh']
X_raw = test_idf[features] #Full dataset with every features


#Base pipeline
base_pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('resampler', SMOTETomek(random_state=143)),
    ('classifier', None)
])

#Classification model definition
classifiers = {
    'xgboost': XGBClassifier(random_state=48, 
                             eval_metric='auc'),
    'logistic_regression': LogisticRegression(random_state=42, 
                                              penalty='elasticnet',
                                              solver='saga'),
    'random_forest': RandomForestClassifier(random_state=42, 
                                            max_samples=0.7),
    'svm': SVC(random_state=42, 
               probability=True)
}

#Parameters search space
search_spaces = {
    'xgboost': {
        'classifier__learning_rate': Real(0.01, 0.2, prior='log-uniform'),
        'classifier__max_depth': Integer(2, 4),
        'classifier__n_estimators': Integer(100, 500),
        'classifier__subsample': Real(0.6, 1.0, 'uniform'),
        'classifier__colsample_bytree': Real(0.6, 1.0, 'uniform'),
        'classifier__gamma': Real(0, 1),
        'resampler__sampling_strategy': Real(0.5, 0.7)
    },
    'logistic_regression': {
        'classifier__C': Real(0.01, 10.0, prior='log-uniform'),
        'classifier__l1_ratio': Real(0.3, 0.7, 'uniform'),
        'resampler__sampling_strategy': Real(0.5, 0.7)
    },
    'random_forest': {
        'classifier__n_estimators': Integer(100, 500),
        'classifier__max_depth': Integer(2, 4),
        'classifier__min_samples_split': Integer(5, 20),
        'classifier__max_features': Categorical(['sqrt', 0.5]),
        'resampler__sampling_strategy': Real(0.5, 0.7)
    },
    'svm': {
        'classifier__C': Real(0.5, 50.0, prior='log-uniform'),
        'classifier__gamma': Categorical(['scale', 'auto'] + list(np.logspace(-3, -1, 5))),
        'classifier__kernel': Categorical(['rbf', 'linear']),
        'resampler__sampling_strategy': Real(0.5, 0.7)
    }
}

final_results = []
for features_set_name, context in config['features_sets'].items():
    X = X_raw[context]
    #X = X_raw[dico_features['Without']]
    #X_train, X_test, y_train, y_test = train_test_split(
     #   X, y, 
      #  test_size=config['test_size'],
       # stratify=y,
        #random_state=42
    #)
    #X_train_selected, X_test_selected, features_selected, best_max_features = select_features(
     #   X_train, X_test, y_train,
      #  max_features_range=(25, 50),
       # n_iter=config['n_iter'])

    #dico_features_tfidf[features_set_name] = features_selected

    df_results = optimized_training(
        X_train_selected, X_test_selected, 
        y_train, y_test,
        classifiers, config, 
        'randomforest', search_spaces
    )

    final_results.append(df_results)

final_df = pd.concat(final_results)
final_df

## Cross Validation (CV=5)

In [ ]:
##Contingency Table of the best model 
#1 y creation 
y = test_idf['med_tresh']

#2 X Creation
X = test_idf[dico_features_tfidf['With']] #Full dataset with every features

X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=config['test_size'],
        stratify=y,
        random_state=42
    )

## Base pipeline
base_pipeline = ImbPipeline([
            ('scaler', StandardScaler()),
            ('resampler', SMOTETomek(sampling_strategy=0.7, random_state=143)),
            ('classifier', XGBClassifier(
                learning_rate=0.01,
                max_depth=10,
                n_estimators=298,
                subsample=0.6804196802009144,
                colsample_bytree=0.6,
                gamma=0.0,
                random_state=48
            ))
        ])

base_pipeline.fit(X_train, y_train)
train_pred = base_pipeline.predict(X_train)
test_pred = base_pipeline.predict(X_test)
train_proba = base_pipeline.predict_proba(X_train)[:, 1]
test_proba = base_pipeline.predict_proba(X_test)[:, 1]
y_pred = base_pipeline.predict(X_test)
train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)
f1 = f1_score(y_test, test_pred)
auc = roc_auc_score(y_test, test_proba)
print(f'test acc: {test_acc}, train acc: {train_acc}, f1: {f1}, auc: {auc}')


## Results analysis

### Confusion matrix

I will look at the results from the confusion for the best model out of 8

In [ ]:
#Create the confusion matrix
cm = confusion_matrix(y_test, y_pred, normalize='true')

#Get the name of the classes rather than 0 and 1
labels = ['Negative', 'Positive']

#Convertion to a dataframe
cm_df = pd.DataFrame(cm, index=[f'Actual {label}' for label in labels],
                         columns=[f'Predicted {label}' for label in labels])

# Palette 1 : Philadelphia 76ers (bleu royal) - with ChatGPT
sixers_colors = ['#E6EFFF', '#A3BFFA', '#0047AB', '#003087']  # Très pâle, pâle, bleu moyen, bleu royal
sixers_cmap = LinearSegmentedColormap.from_list("Sixers", sixers_colors)

#Display the matrix
def plot_confusion_matrix(cm_df, cmap, title):
    plt.figure(figsize=(8, 6), dpi=360)
    sns.heatmap(cm_df, annot=True, fmt='.2f', cmap=cmap, cbar=True,
                annot_kws={'size': 12, 'weight': 'bold'}, linewidths=0.5, 
                linecolor='white')
    plt.title(title, fontsize=14, pad=15)
    plt.xlabel('Predictions', fontsize=12)
    plt.ylabel('Real Values', fontsize=12)
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(cm_df, sixers_cmap, 'Confusion Matrix from XGBoost - Norm true')

### Shapley values

I will look at the shapley values for the best model 

In [ ]:
#Do not work with ImbPipeline, hence we redo the train manually
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)
sm = SMOTETomek(sampling_strategy = 0.7, random_state=143)
X_resampled, y_resampled = sm.fit_resample(X_train_scaled, y_train)

model = XGBClassifier(learning_rate=0.01,
                        max_depth=7,
                        n_estimators=298,
                        subsample=0.6804196802009144,
                        colsample_bytree=0.6,
                        gamma=0.0,
                        random_state=48)
model.fit(X_resampled, y_resampled)

#Shap values calculation
explainer = shap.Explainer(model, X_resampled)
shap_values = explainer(X_test_scaled_df)

#Beeswarm plot
shap.plots.beeswarm(shap_values, max_display=10)  #10 to keep lisibility

### Do we improve GMs decisions ? 

In [ ]:
player_names = X_test.index

#Get a df with the results from the prediction (of the test set)
results = pd.DataFrame({
    'Player': player_names,
    'Predicted': y_pred,
    'Actual': y_test
})

results['Results'] = results['Predicted'] == results['Actual'] #
results

In [ ]:
results[results['Results'] == False]